In [ ]:
import os
import sys, importlib

In [ ]:
import torch

def detect_environment():
    # Check for GPU
    has_gpu = torch.cuda.is_available()

    # Check for Colab-specific environment
    is_colab = 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython())

    # Check for RunPod-specific environment
    is_runpod = 'RUNPOD_POD_ID' in os.environ or os.path.exists('/workspace')

    if is_runpod and has_gpu:
        return 'runpod'
    elif is_colab and not has_gpu:
        return 'colab'
    else:
        return 'unknown'

In [ ]:
from pathlib import Path
####cxr-notebooks CODE
####foundation-models-radiology - CXR images

env = detect_environment()
print("env", env)
if env == 'runpod':
  ##rclone sync gdrive:/MyDrive/MLProjects/foundation-models-radiology /workspace/MLProjects/foundation-models-radiology
  ROOT = Path('/workspace/MLProjects/foundation-models-radiology')
  UTILS_DIR = '/workspace/MLProjects/cxr-notebooks'
elif env == 'colab':
  from google.colab import drive
  drive.mount('/content/drive')
  ###once mounted the folders can be referenced
  ROOT = Path('/content/drive/MyDrive/MLProjects/foundation-models-radiology')
  UTILS_DIR = '/content/drive/MyDrive/MLProjects/cxr-notebooks'
else:
  sys.exit("Error: No platform recognised")
  UTILS_DIR = str(Path.cwd())

sys.path.append(UTILS_DIR)

DICOM_DIR = ROOT / 'PTXHeadtoHeadSmall'   # use the exact folder name as on Drive
JPEG_DIR = ROOT / 'cxr_jpegs'
JPEG_DIR.mkdir(exist_ok=True)
print("exists:", ROOT.exists())


In [ ]:
from pathlib import Path
import sys

UTILS_DIR = Path("/workspace/MLProjects/cxr-notebooks")  # <- set this correctly
print("cwd:", os.getcwd())
print("on sys.path?", str(UTILS_DIR) in sys.path)
print("exists:", UTILS_DIR.exists())
print("utils.py exists:", (UTILS_DIR / "utils.py").exists())
print("entries:", [p.name for p in UTILS_DIR.iterdir()])

sys.path.insert(0, str(UTILS_DIR))  # put utils_dir at the front of sys.path

In [ ]:
from importlib.util import spec_from_file_location, module_from_spec
from pathlib import Path

UTILS_DIR = Path("/workspace/MLProjects/cxr-notebooks")
utils_path = UTILS_DIR / "utils.py"   # change to 'utils/__init__.py' if it's a package folder
assert utils_path.exists(), f"Not found: {utils_path}"

spec = spec_from_file_location("utils", str(utils_path))
utils = module_from_spec(spec)
spec.loader.exec_module(utils)

print("utils loaded from:", utils.__file__)
print(utils.ping())  # whatever function you expect

In [ ]:
from zipfile import ZipFile
from pathlib import Path

DEST = Path(ROOT)
zip_files = list(DEST.glob("*.zip"))
if not zip_files:
    raise FileNotFoundError("No zip files found in folder.")
zip_path = zip_files[0]
#zip_path = next(DEST.glob("*.zip"))  # first zip in the folder
with ZipFile(zip_path) as zf:
    zf.extractall(ROOT)
#zip_path.unlink()  # delete the zip

In [ ]:
from pathlib import Path

# ---- Example call (your paths)
# Source DICOMs here:
#DICOM_DIR = ROOT / 'PTXHeadtoHeadSmall'   # use the exact folder name as on Drive
#JPEG_DIR = ROOT / 'cxr_jpegs'
#src = "/workspace/MLProjects/PTXHeadtoHeadSmall"
# JPEGs out here:
#dst = "/workspace/MLProjects/PTXHeadtoHeadSmall/cxr_jpegs"
#utils.make_destdir() - actually made above

dicoms = utils.loop_files(DICOM_DIR, "*.dcm", recursive=True) ##returns a dicom paths array
valid_dicoms = utils.check_return_filetypes(dicoms, "dcm")

utils.convert_dicom_to_jpegs(valid_dicoms, JPEG_DIR, resize_short=1024, quality=95, save_png=True)


In [ ]:
pngs = utils.loop_files(JPEG_DIR, "*.png", recursive=True)
valid_pngs = utils.check_return_filetypes(pngs, "png")
print(valid_pngs)
print(len(valid_pngs))


In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
###example config
# Configure LoRA: adapt attention projections (common choice)
#config = LoraConfig(
#    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
#    target_modules=["query", "key", "value", "dense"]  # adjust per arch
#)
#model = get_peft_model(model, config)
#model.print_trainable_parameters()  # sanity check: tiny % trainable
!pip install pandas

#!pip install bitsandbytes
#!pip install accelerate

import torch, transformers, accelerate, bitsandbytes as bnb
print("torch", torch.__version__, "cuda", torch.version.cuda, "gpu?", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("accelerate", accelerate.__version__)
print("bitsandbytes", bnb.__version__)


In [ ]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
from PIL import Image
import warnings, re
import requests
from datetime import datetime
from io import BytesIO
from accelerate import Accelerator

# ---- Put cache/temp on /workspace (adjust if needed)
os.environ.setdefault("HF_HOME", "/workspace/.hf")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.environ.setdefault("TRANSFORMERS_CACHE", "/workspace/.hf/transformers")
os.makedirs(os.environ["TRANSFORMERS_CACHE"], exist_ok=True)
os.environ.setdefault("TMPDIR", "/workspace/tmp")
os.makedirs(os.environ["TMPDIR"], exist_ok=True)
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

# ---- Small helpers
def _stamp(t0=None):
    wall = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    if t0 is None:
        return f"[{wall}]"
    return f"[{wall} +{time.perf_counter()-t0:.2f}s]"

def load_image_from_url(url, timeout=(10, 60)):
    r = requests.get(url, timeout=timeout); r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")

class CheXagent(object):
    def __init__(self):
        self.t0 = time.perf_counter()

        # step 1: Setup constant
        self.model_name = "StanfordAIMI/CheXagent-2-3b"
        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        ##ver 10.6
        torch.cuda.set_per_process_memory_fraction(0.9, device=0)
        
        major, minor = (torch.cuda.get_device_capability(0) if self.device.type=="cuda" else (0,0)) #added
        self.supports_bf16 = (major >= 8) #added

        self.dtype      = torch.bfloat16 if self.device.type == "cuda" else torch.float32
        device_map      = "cuda:0" if self.device.type == "cuda" else None
        ###self.revision is the exact snapshot of the model repo on Hugging Face that you want to load.
        ####You pass it to from_pretrained(..., revision=self.revision) so Transformers fetches the files (weights + “remote code” like modeling_chexagent.py, tokenization_chexagent.py) from that specific commit/tag/branch.
        self.revision   = "463999422d77fe01380ef03493e5ae3bb11bd004"  # pin remote code

        # step 2: Load Processor and Model
        print(_stamp(self.t0), "loading tokenizer…", flush=True)
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name, trust_remote_code=True, revision=self.revision,
            cache_dir=os.environ.get("TRANSFORMERS_CACHE")
        )
        print(_stamp(self.t0), "loading model (this can take a while first run)…", flush=True)
        # 1) Load on CPU in fp32 (safe), no sharding
        base = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            trust_remote_code=True, revision=self.revision,
            cache_dir=os.environ.get("TRANSFORMERS_CACHE"),
            device_map=None,
            torch_dtype=torch.float32
        )
        ##1011 max_memory={0: "40GiB", "cpu": "32GiB"}, ##low_cpu_mem_usage=False,    
        # Optional: turn off KV cache in training
        base.config.use_cache = False
        
        # 2) If CUDA is available, 
        self.model = base.to(self.device)  # got rid of 2nd argument - dtype=torch.bfloat16 (keep weights as loaded (fp32))
        if self.device.type == "cuda":
            torch.backends.cuda.matmul.allow_tf32 = True
        self.model.eval()
        print(_stamp(self.t0), "model ready.", flush=True)

    def generate(self, paths, prompt):
        t0 = time.time()
        print("[step 1] before tokenizer", flush=True)
        query = self.tokenizer.from_list_format(
            [*({'image': str(p)} for p in paths), {'text': prompt}]
        )
        print("[step 2] after tokenizer, before apply_chat_template", flush=True)

        ##added .to(self.device) find out what difference
        conv = [{"from": "system", "value": "You are a helpful assistant."}, {"from": "human", "value": query}]
        input_ids = self.tokenizer.apply_chat_template(
            conv, add_generation_prompt=True, return_tensors="pt"
        ).to(self.device)

        attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=self.device)

        output = self.model.generate(
            input_ids.to(self.device), attention_mask=attention_mask, do_sample=False, num_beams=1, temperature=1., top_p=1., use_cache=True,
            max_new_tokens=512
        )[0]
        print("[step 3] after appy_chat_template", flush=True)
        response = self.tokenizer.decode(output, skip_special_tokens=True)
        ##original - but prev. a problem -> response = self.tokenizer.decode(output[input_ids.size(1):-1])
        return response

    def view_classification(self, path):
        assert isinstance(path, str)
        prompt = "What is the view of this chest X-ray? Options: (a) PA, (b) AP, (c) LATERAL"
        response = self.generate([path], prompt)
        return response

    ##passing src
    def binary_disease_classification(self, src, disease_name, max_new_tokens=256, max_images=3):
        ##plan to move this to its own function
        pngs = utils.loop_files(src, "*.png", recursive=True)
        valid_pngs = utils.check_return_filetypes(pngs, "png")
        ###end plan
        prompt = f'Does this chest X-ray contain a {disease_name}?'
        results = []
        for i, p in enumerate(valid_pngs[:max_images], 1):
            ans = self.generate([p], prompt)
            results.append({"picture": i, "image": p, "answer": ans})
        return results

    def findings_generation(self, src, indication, max_images=3):
        ##plan to move this to its own function at the momement this iterates through the jpegs..
        ##need to iterate through the .dcm's and pass the result into self.generate..
        pngs = utils.loop_files(src, "*.png", recursive=True)
        valid_pngs = utils.check_return_filetypes(pngs, "png")
        ###end plan
        prompt = f'Given the indication: "{indication}", write a structured findings section for the CXR.'
        results = []
        for i, p in enumerate(valid_pngs[:max_images], 1):
            ans = self.generate([p], prompt)
            results.append({"picture": i, "image": p, "answer": ans})
        return results

    def findings_generation_section_by_section(self, paths):
        assert isinstance(paths, list)
        anatomies = [
            "Airway", "Breathing", "Cardiac",
            "Diaphragm",
            "Everything else (e.g., mediastinal contours, bones, soft tissues, tubes, valves, and pacemakers)"
        ]
        prompts = [f'Please provide a detailed description of "{anatomy}" in the chest X-ray' for anatomy in anatomies]
        responses = []
        for anatomy, prompt in zip(anatomies, prompts):
            response = self.generate(paths, prompt)
            responses.append((anatomy, response))
        return responses

    # ----------------- LoRA fine-tuning -----------------
    def add_lora(self, r=8, lora_alpha=32, lora_dropout=0.1):
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
            target_modules=["q_proj","k_proj","v_proj"]
        )
        #target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
        # wrap the CURRENT model (also fixes self.base not being set)
        self.model = get_peft_model(self.model, lora_config)
        self.model.print_trainable_parameters()

    # ----------------- Dataset generator for Trainer -----------------
    def _dataset_from_csv(self, csv_path, image_dir, prompt):
        df = pd.read_csv(csv_path, names=["image","label"])
        examples = []

        ##for _, row in df.iterrows():
        for i in range(len(df)):
            path, lab = df.iloc[i].image, int(df.iloc[i].label)
            tgt = "yes" if lab == 1 else "no"

            # Make sure path is string
            img_path = str(image_dir / path)
            query_obj = self.tokenizer.from_list_format([{"image": img_path}, {"text": prompt}])
            ##query_str = str(query_obj)
            
            ##two different prefix encodings: This is the prefix without the assistant role/content from full_conv.
            pref_conv = [
                {"from":"system","value":"You are a helpful assistant."},
                {"from":"human","value":query_obj}  #####note: pass the object, not str()
            ]
            full_conv = pref_conv + [{"from": "assistant", "value": tgt}]

            # Correct prefix encoding (assistant role + no answer)
            prefix_ids = self.tokenizer.apply_chat_template(
                pref_conv, add_generation_prompt=True, return_tensors="pt"
            )[0]

            # Full sequence including assistant answer
            full_ids = self.tokenizer.apply_chat_template(
                full_conv, add_generation_prompt=False, return_tensors="pt"
            )[0]

            labels = torch.full_like(full_ids, -100)
            answer_start = len(full_ids) - len(self.tokenizer(tgt, return_tensors="pt")["input_ids"][0])
            labels[answer_start:] = full_ids[answer_start:]
            
            #labels = full_ids.clone() 
            #labels[:prefix_ids.shape[0]] = -100 ##prefix_ids contains only the system + human part This sets the first N tokens in labels to -100, where N is the length of the prompt. (“Ignore these during training.”)
            print("Valid label tokens:", (labels != -100).sum().item()) ##should see a small number (typically 1–3), confirming that only the assistant’s answer is being trained.
            
            attn = torch.ones_like(full_ids, dtype=torch.long) ##attention mask of all ones — meaning every token is attended to (no padding yet).
            
            print("full_ids:", full_ids.shape[0])
            print("Decoded answer:", self.tokenizer.decode(full_ids[answer_start:]))
            
            examples.append({
                "input_ids": full_ids,
                "labels": labels,
                "attention_mask": attn
            })
            
        return examples

    # ----------------- Collator -----------------
    @staticmethod
    def collate(batch, pad_id=None):
        maxlen = max(x["input_ids"].shape[0] for x in batch)
        out = {}
        for k in ("input_ids","labels","attention_mask"):
            items = []
            for x in batch:
                t = x[k]; pad = maxlen - t.shape[0]
                if pad > 0:
                    if k == "labels":
                        t = torch.cat([t, torch.full((pad,), -100, dtype=t.dtype)])
                    elif k == "attention_mask":
                        t = torch.cat([t, torch.zeros(pad, dtype=torch.long)])
                    else:
                        t = torch.cat([t, torch.full((pad,), pad_id, dtype=torch.long)])
                items.append(t)
            out[k] = torch.stack(items, 0)
        return out  
    
    # ----------------- LoRA training -----------------
    def train_lora(self, csv_path, image_dir, prompt, output_dir, batch_size=1, lr=2e-4, epochs=10):
        # Make sure LoRA is added
        #if not hasattr(self.model, "active_adapters"): commented 11/10
        self.add_lora()
        ##not valid - self.gradient_checkpointing_enable() ##This trades compute for memory — great for large models.
        #if hasattr(self.model, "gradient_checkpointing_enable"): commented 11/10
        #    self.model.gradient_checkpointing_enable() commented 11/10
        # Dataset
        dataset = self._dataset_from_csv(csv_path, image_dir, prompt)

        print("=== DATASET DEBUG ===")
        ex = dataset[0]
        print("input_ids.shape:", ex["input_ids"].shape)
        print("labels.shape:", ex["labels"].shape)
        print("attention_mask.shape:", ex["attention_mask"].shape)
        
        num_valid = (ex["labels"] != -100).sum().item()
        print("num valid label tokens:", num_valid)
        print("first 40 labels:", ex["labels"][:40].tolist())

        # Check LoRA actually attached & only adapters are trainable
        self.model.print_trainable_parameters()

        # TrainingArguments
        args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=batch_size,
            num_train_epochs=epochs,
            learning_rate=lr,
            bf16=(self.device.type=="cuda" and self.supports_bf16),
            fp16=(self.device.type=="cuda" and not self.supports_bf16),
            logging_steps=1,
            report_to="none",
            save_strategy="no", 
            remove_unused_columns=False  # <-- important for custom inputs added 11/10
        )
        pad_id = self.tokenizer.pad_token_id or self.tokenizer.eos_token_id

        # Trainer
        trainer = Trainer(
            model=self.model,
            args=args,
            train_dataset=dataset,
            data_collator=lambda b: self.collate(b, pad_id=pad_id)
        )
        trainer.train()
                
        # Save LoRA + tokenizer
        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)   


In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


In [ ]:
from rich import print
# Make sure caches & temp are on /workspace, and disable hf-xet just in case

def main():
    # Load the model
    os.environ["HF_HOME"] = "/workspace/.hf"
    os.environ["TRANSFORMERS_CACHE"] = "/workspace/.hf/transformers"
    os.environ["TMPDIR"] = "/workspace/tmp"
    os.environ["HF_HUB_DISABLE_XET"] = "1"
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

    chexagent = CheXagent()

    try:
        # Task 3: Binary Disease Classification
        #path = "/workspace/MLProjects/PTXHeadtoHeadSmall/cxr_jpegs"
        #response = chexagent.binary_disease_classification(path, "Pneumothorax")
        resp = chexagent.binary_disease_classification(
            JPEG_DIR, "Pneumothorax", max_images=6
        )
        for r in resp:
            print(r["picture"], r["answer"])

        #indication = "Shortness of breath and mild left sided pleuritic chest pain. No trauma."
        #respF = chexagent.findings_generation(
        #    JPEG_DIR, "indication", max_images=2
        #)
        #for rf in respF:
        #    print(rf["picture"], rf["answer"])


    except Exception as e:
        print("[ERROR]", repr(e))
        traceback.print_exc()   # <— show the real stack trace
        raise                   # <— let it surface

if __name__ == '__main__':
    main()

In [ ]:
#csv_path → CSV of image paths + 0/1 labels
#image_dir → folder containing the PNGs
#prompt → question like "Does this chest X-ray contain a pneumothorax?"
#output_dir → where to save LoRA weights + tokenizer
#def train_lora(self, csv_path, image_dir, prompt, output_dir, batch_size=1, lr=2e-4, epochs=1):

import torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

MODEL_ID   = "StanfordAIMI/CheXagent-2-3b"
ROOT = Path('/workspace/MLProjects/foundation-models-radiology')
IMG_PATH = ROOT / "cxr_jpegs"
#ROOT = Path('/content/drive/MyDrive/MLProjects/foundation-models-radiology')
CSV_PATH   = ROOT / "ground-truth.csv"   # two cols: image,label
OUT_DIR    = ROOT / "chex_lora_adapter"
PROMPT     = "Does this chest X-ray contain a pneumothorax? Answer yes or no."

chexagent = CheXagent()
try:
    # Task 3: Binary Disease Classification
    #path = "/workspace/MLProjects/PTXHeadtoHeadSmall/cxr_jpegs"
    #response = chexagent.binary_disease_classification(path, "Pneumothorax")
    ##def train_lora(self, csv_path, image_dir, prompt, output_dir, batch_size=1, lr=2e-4, epochs=1):
    chexagent.train_lora(
        CSV_PATH, IMG_PATH, PROMPT,  OUT_DIR
    )
except Exception as e:
    print("[ERROR]", repr(e))
    #traceback.print_exc()   # <— show the real stack trace
    raise 

In [ ]:
resp = chexagent.binary_disease_classification(
    JPEG_DIR, "Pneumothorax", max_images=6
)
for r in resp:
    print(r["picture"], r["answer"])


In [ ]:

!pip install open_clip_torch==2.23.0 matplotlib
import pydicom
import matplotlib.pyplot as plt
import time
from datetime import datetime

# ---- Small helpers
def _stamp(t0=None):
    wall = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    if t0 is None:
        return f"[{wall}]"
    return f"[{wall} +{time.perf_counter()-t0:.2f}s]"

class BioMedClip(object):
    def __init__(self):
        self.t0 = time.perf_counter()

        ####BiomedCLIP
        from open_clip import create_model_from_pretrained, get_tokenizer # works on open-clip-torch>=2.23.0, timm>=0.9.8

        model, preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
        tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')

        ##similar to base = AutoModelForCausalLM.from_pretrained for CheXagent
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = model.to(device).eval()

        print(_stamp(self.t0), "model ready.", flush=True)

    def generate(self, paths, prompt):
        t1 = time.time()
        ##tokeniser pass in paths and prompts
        ##chat_template
        ##does not appear to have the same need for generate as CheXagent
        #output = self.model.generate(
        #    input_ids.to(self.device), attention_mask=attention_mask, do_sample=False, num_beams=1, temperature=1., top_p=1., use_cache=True,
        #    max_new_tokens=512
        #)[0]

    ##JPEG_DIR, "Pneumothorax", max_images=3 from binary disease classification - Zero shot in CheXagent
    def binary_disease_classification(self, src, disease_name, max_new_tokens=256, max_images=3):
        t2 = time.time()
        #dicom_paths = glob.glob(str(ROOT / 'PTX Head to Head Study Data' / '**/*.dcm'), recursive=True)
        #images = torch.stack([load_dicom_image(path,preprocess) for path in dicom_paths]).to(device)



In [ ]:
import numpy as np

biomed = BioMedClip()

dicom_paths = utils.loop_files(DICOM_DIR, "*.dcm", recursive=True)
valid_dicoms = utils.check_return_filetypes(dicoms, "dcm")

###from ....
dicom_file = pydicom.dcmread(valid_dicoms[2])
img_array = dicom_file.pixel_array.astype(np.float32) ##just casting to float32 (numpy array)
img_array.shape ##(H, W) single-frame grayscale (most CXRs), (N, H, W): multi-frame, (H, W, 3/4): color
plt.imshow(img_array)

##biomed.model.train ##- where does this come from?


In [ ]:
####this is what is done when dicoms are converted to .png/.jpgs for the CheXagent code
from torchvision import transforms
from PIL import Image

img_array -= img_array.min()
img_array /= img_array.max()
img_array *= 255
img_array = img_array.astype(np.uint8)

plt.imshow(img_array)



In [ ]:
if len(img_array.shape) == 2:  # grayscale
    img = Image.fromarray(img_array).convert("RGB")
elif len(img_array.shape) == 3 and img_array.shape[2] == 3:
    img = Image.fromarray(img_array)
else:
    raise ValueError(f"Unsupported DICOM image shape: {img_array.shape}")
plt.imshow(img)

other_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
])
processed_img = other_preprocess(img)
plt.imshow(processed_img.permute(1, 2, 0).cpu().numpy())

In [ ]:
# --- tokenizer & base model ---
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token is None and tok.eos_token is not None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"


In [ ]:
##similar to the original
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
base.config.use_cache = False


In [ ]:
# --- LoRA ---
lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32, lora_dropout=0.1, bias="none",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(base, lora_config)
############## sanity check: tiny % trainable ########################
model.print_trainable_parameters()


In [ ]:
# --- tiny dataset from CSV ---
df = pd.read_csv(CSV_PATH, names=["image","label"])
print(CSV_PATH)
print(JPEG_DIR)

In [ ]:

class DS(torch.utils.data.Dataset):
    def __len__(self): return len(df)
    def __getitem__(self, i):
        path, lab = str(df.iloc[i].image), int(df.iloc[i].label)
        tgt = "yes" if lab == 1 else "no"
        img_path = str(JPEG_DIR / path)   # ensure string, not Path
        query = tok.from_list_format([{"image": img_path}, {"text": PROMPT}])
        #query_str = str(query) 
        pref_conv = [{"from":"system","value":"You are a helpful assistant."},
                     {"from":"human","value":query}]
        pref_ids = tok.apply_chat_template(pref_conv, add_generation_prompt=True, return_tensors="pt")[0]
        full_conv = pref_conv + [{"from":"assistant","value":tgt}]
        full_ids = tok.apply_chat_template(full_conv, add_generation_prompt=False, return_tensors="pt")[0]
        labels = full_ids.clone(); labels[:len(pref_ids)] = -100
        attn = torch.ones_like(full_ids, dtype=torch.long)
        return {"input_ids": full_ids, "labels": labels, "attention_mask": attn}
ds = DS()

# simple right-padding collator
##batch["input_ids"] = batch["input_ids"].to(torch.bfloat16)
def collate(batch):
    maxlen = max(x["input_ids"].shape[0] for x in batch)
    pad_id = tok.pad_token_id or tok.eos_token_id
    out = {}
    for k in ("input_ids","labels","attention_mask"):
        items = []
        for x in batch:
            t = x[k]; pad = maxlen - t.shape[0]
            if pad > 0:
                if k == "labels":         t = torch.cat([t, torch.full((pad,), -100, dtype=torch.long)])
                elif k == "attention_mask": t = torch.cat([t, torch.zeros(pad, dtype=torch.long)])
                else:                     t = torch.cat([t, torch.full((pad,), pad_id, dtype=torch.long)])
            items.append(t)
        out[k] = torch.stack(items, 0)
    return out

# --- train quickly (1 epoch over 5 samples) ---
args = TrainingArguments(
    output_dir=OUT_DIR, per_device_train_batch_size=1,
    learning_rate=2e-4, num_train_epochs=1, logging_steps=1,
    bf16=torch.cuda.is_available(), fp16=False, report_to="none", save_strategy="no"
)
Trainer(model=model, args=args, train_dataset=ds, data_collator=collate).train()

# --- save the LoRA adapter ---
model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)



In [ ]:
import torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "StanfordAIMI/CheXagent-2-3b"
REVISION = "463999422d77fe01380ef03493e5ae3bb11bd004"

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, revision=REVISION)
if tok.pad_token is None and tok.eos_token is not None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,                 # <- NOT load_in_8bit
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
)

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, revision=REVISION,
    quantization_config=bnb_cfg, device_map="auto"
)
base.config.use_cache = False
base.gradient_checkpointing_enable()
base = prepare_model_for_kbit_training(base)

# LoRA adapter on decoder
lcfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32, lora_dropout=0.1, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
)
model = get_peft_model(base, lcfg)

# --- tiny dummy dataset just to confirm training runs ---
texts = ["hello lora", "peft makes finetuning small"]
enc = tok(texts, return_tensors="pt", padding=True)
enc["labels"] = enc["input_ids"].clone()

args = TrainingArguments(
    output_dir="out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    bf16=torch.cuda.is_available(),
    optim="paged_adamw_8bit",          # memory-friendly optimizer (needs accelerate/bnb)
    logging_steps=1,
    report_to="none",
    save_strategy="no",
)

Trainer(model=model, args=args, train_dataset=torch.utils.data.TensorDataset(
    enc["input_ids"], enc["attention_mask"], enc["labels"]
), data_collator=lambda batch: {
    "input_ids": torch.nn.utils.rnn.pad_sequence([b[0] for b in batch], batch_first=True, padding_value=tok.pad_token_id),
    "attention_mask": torch.nn.utils.rnn.pad_sequence([b[1] for b in batch], batch_first=True, padding_value=0),
    "labels": torch.nn.utils.rnn.pad_sequence([b[2] for b in batch], batch_first=True, padding_value=-100),
}).train()

model.save_pretrained("chex_lora_adapter")
tok.save_pretrained("chex_lora_adapter")


In [ ]:
# --- load adapter for inference & test on one image ---
##model = get_peft_model(base, lora_config) **big difference is this is above (before LoRA) and
##model = PeftModel.from_pretrained(base, OUT_DIR).eval() - this runs on the OUT_DIR where the LoRA model is stored..

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
).eval()
model = PeftModel.from_pretrained(base, OUT_DIR).eval()

def predict(img_path: str) -> str:
    q = tok.from_list_format([{"image": img_path}, {"text": PROMPT}])
    conv = [{"from":"system","value":"You are a helpful assistant."},
            {"from":"human","value": q}]
    ids = tok.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(**{"input_ids": ids}, max_new_tokens=3, do_sample=False)
    txt = tok.decode(out[0], skip_special_tokens=True).lower()
    return "yes" if "yes" in txt.split()[:3] else "no"

print("Pred(0):", predict(str(df.iloc[0].image)))

In [ ]:
##Low-Rank Adaptation (LoRA) is a very common PEFT method that decomposes the weight matrix into two smaller trainable matrices. Start by defining a LoraConfig object with the parameters shown below.
##python -m pip install --no-cache-dir "transformers==4.40.0" "peft==0.11.1"  in Startup script
#from peft import LoraConfig, TaskType, get_peft_model
#from transformers import AutoModelForCausalLM

# create LoRA configuration object
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # type of task to train on
    inference_mode=False, # set to False for training
    r=8, # dimension of the smaller matrices
    lora_alpha=32, # scaling factor
    lora_dropout=0.1 # dropout of LoRA layers
)
# Wrap the base model so PEFT (parameter effecient fine tuning) methods (and parameter freezing) are in place
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # sanity check: tiny % trainable

#model.add_adapter(lora_config, adapter_name="lora_1") — adds a LoRA adapter named "lora_1" into the model’s target Linear layers (q/k/v/o proj, etc.).
model.add_adapter(lora_config, adapter_name="lora_1")

# --- tiny toy dataset ---
texts = ["hello lora", "peft makes finetuning small"]
enc = tokenizer(texts, return_tensors="pt", padding=True)
enc["labels"] = enc["input_ids"].clone()

args = TrainingArguments(
    output_dir="out",
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=torch.cuda.is_available(),
    logging_steps=1,
)
trainer = Trainer(model=model, args=args, train_dataset=enc)
trainer.train()


In [ ]:
# --- NEW IMPORTS FOR TRAINING ---
from dataclasses import dataclass
from typing import List, Optional, Union
from peft import LoraConfig, TaskType, get_peft_model
from transformers import Trainer, TrainingArguments

# ========= Add these helpers to your module =========
@dataclass
class DynamicPadCollator:
    pad_id: int
    def __call__(self, batch):
        max_len = max(x["input_ids"].size(0) for x in batch)
        input_ids, labels, attn = [], [], []
        for x in batch:
            ids, labs, mask = x["input_ids"], x["labels"], x["attention_mask"]
            pad = max_len - ids.size(0)
            if pad > 0:
                ids  = torch.cat([ids,  torch.full((pad,), self.pad_id, dtype=torch.long)])
                labs = torch.cat([labs, torch.full((pad,), -100,   dtype=torch.long)])
                mask = torch.cat([mask, torch.zeros(pad,           dtype=torch.long)])
            input_ids.append(ids); labels.append(labs); attn.append(mask)
        return {
            "input_ids": torch.stack(input_ids),
            "labels": torch.stack(labels),
            "attention_mask": torch.stack(attn),
        }


class CheXJsonDataset(torch.utils.data.Dataset):
    """
    records: list of dicts like:
      {
        "image": "/abs/path.jpg" or ["/a.jpg", "/b.jpg"],
        "instruction": "What abnormalities...",
        "answer": "There is ..."
      }
    We build a single-turn chat:
      system: "You are a helpful assistant."
      human : tokenizer.from_list_format([{'image': path(s)}, {'text': instruction}])
      assistant: answer
    labels = full_ids with prompt tokens masked to -100.
    """
    def __init__(self, records: List[dict], tokenizer, device: torch.device):
        self.records = records
        self.tok = tokenizer
        self.device = device
        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token

    def __len__(self): return len(self.records)

    def _ensure_paths(self, img: Union[str, List[str]]) -> List[str]:
        if isinstance(img, str):
            return [img]
        elif isinstance(img, (list, tuple)):
            return [str(p) for p in img]
        raise TypeError(f"image must be str or list[str], got {type(img)}")

    def __getitem__(self, i):
        ex = self.records[i]
        paths = self._ensure_paths(ex["image"])
        instruction = ex["instruction"]
        answer = ex["answer"]

        # Build the human message with images + instruction
        query = self.tok.from_list_format(
            [*({'image': p} for p in paths), {'text': instruction}]
        )

        # 1) Prompt-only ids (system + human + assistant prefix) to get the cut point
        ###NEED AGAIN IS THERE A PTX?
        conv_prompt = [
            {"from": "system", "value": "You are a helpful assistant."},
            {"from": "human",  "value": query},
        ]
        prompt_ids = self.tok.apply_chat_template(
            conv_prompt, add_generation_prompt=True, return_tensors="pt"
        )[0]  # shape (seq,)

        # 2) Full conversation including assistant answer
        conv_full = [
            {"from": "system",    "value": "You are a helpful assistant."},
            {"from": "human",     "value": query},
            {"from": "assistant", "value": answer},
        ]
        full_ids = self.tok.apply_chat_template(
            conv_full, add_generation_prompt=False, return_tensors="pt"
        )[0]  # shape (seq,)

        # Labels = full_ids with prompt tokens masked to -100
        labels = full_ids.clone()
        cut = prompt_ids.size(0)
        labels[:cut] = -100

        attention_mask = torch.ones_like(full_ids, dtype=torch.long)
        return {
            "input_ids": full_ids.to(torch.long),
            "labels": labels.to(torch.long),
            "attention_mask": attention_mask,
        }

In [ ]:
# ========= Extend your existing CheXagent class with LoRA + training =========

def enable_lora(self,
                r: int = 8,
                alpha: int = 32,
                dropout: float = 0.1,
                target_modules: Optional[List[str]] = None,
                modules_to_save: Optional[List[str]] = None,
                adapter_name: str = "lora_chex"):
    """
    Wrap self.model with a PEFT LoRA adapter.
    By default, we adapt attention projections. You can customize target_modules if needed.
    """
    # Auto target modules if none provided
    if target_modules is None:
        # Probe module names to choose common projection layers
        probe = set()
        for n, m in self.model.named_modules():
            if isinstance(m, torch.nn.Linear):
                leaf = n.split(".")[-1]
                probe.add(leaf)
        # Favor common names seen in your logs: q_proj, k_proj, v_proj, out_proj
        candidates = ["q_proj", "k_proj", "v_proj", "out_proj"]
        target_modules = [c for c in candidates if c in probe]
        if not target_modules:
            # Fallback: adapt all Linear layers named 'q_proj'/'o_proj' variants
            target_modules = ["q_proj", "k_proj", "v_proj", "out_proj"]

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=target_modules,
        modules_to_save=modules_to_save,  # e.g., ["lm_head"] if you want to keep it trainable
        bias="none",
    )
    self.model = get_peft_model(self.model, lora_cfg)
    self.model.print_trainable_parameters()
    return self

##called below from the main class call
#trainer = chex.fit(train_records, val_records=None,
#                   output_dir="chexagent-lora",
#                   num_train_epochs=1, lr=2e-4,
#                   per_device_batch_size=1, grad_accum_steps=8)
def fit(self,
        train_records: List[dict],
        val_records: Optional[List[dict]] = None,
        output_dir: str = "chexagent-lora",
        num_train_epochs: int = 1,
        lr: float = 2e-4,
        per_device_batch_size: int = 1,
        grad_accum_steps: int = 8,
        warmup_ratio: float = 0.03,
        logging_steps: int = 10,
        save_steps: int = 200,
        eval_steps: int = 200):
    """
    Fine-tune with LoRA using HF Trainer.
    train_records / val_records: list of {"image": path or [paths], "instruction": str, "answer": str}
    """
    assert hasattr(self.model, "base_model") or any("lora" in n for n, _ in self.model.named_parameters()), \
        "Call enable_lora() before fit()."

    train_ds = CheXJsonDataset(train_records, self.tokenizer, self.device)
    val_ds   = CheXJsonDataset(val_records,  self.tokenizer, self.device) if val_records else None

    collator = DynamicPadCollator(pad_id=self.tokenizer.pad_token_id)

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_train_epochs,
        learning_rate=lr,
        per_device_train_batch_size=per_device_batch_size,
        per_device_eval_batch_size=per_device_batch_size,
        gradient_accumulation_steps=grad_accum_steps,
        warmup_ratio=warmup_ratio,
        logging_steps=logging_steps,
        save_steps=save_steps,
        evaluation_strategy="steps" if val_ds is not None else "no",
        eval_steps=eval_steps if val_ds is not None else None,
        save_total_limit=2,
        remove_unused_columns=False,   # important for custom dicts
        bf16=(self.device.type == "cuda"),  # train in bf16 on CUDA
        fp16=False,                    # leave False when bf16=True
        report_to=[],
    )

    self.model.train()
    trainer = Trainer(
        model=self.model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
    )
    trainer.train()
    # Save LoRA adapter
    trainer.model.save_pretrained(output_dir)
    # Also keep tokenizer
    self.tokenizer.save_pretrained(output_dir)
    return trainer

# Attach methods to your class
CheXagent.enable_lora = enable_lora
CheXagent.fit = fit

In [ ]:
chex = CheXagent()  # your current constructor (loads base model on GPU bf16 / CPU fp32)

# 1) Enable LoRA (adjust r/alpha/dropout and targets if you like)
chex.enable_lora(r=8, alpha=32, dropout=0.1)

# 2) Your records (file paths preferred; multiple images per example are fine)
train_records = [
    {
      "image": "/workspace/MLProjects/PTXHeadtoHeadSmall/cxr_jpegs/abc.jpg",
      "instruction": "What abnormalities are present?",
      "answer": "No acute cardiopulmonary abnormality."
    },
    {
      "image": ["/path/one.jpg", "/path/two.jpg"],  # multi-image example
      "instruction": "Summarize findings.",
      "answer": "Normal cardiomediastinal silhouette. No focal consolidation or effusion."
    },
]

# 3) Train
trainer = chex.fit(train_records, val_records=None,
                   output_dir="chexagent-lora",
                   num_train_epochs=1, lr=2e-4,
                   per_device_batch_size=1, grad_accum_steps=8)

# 4) Inference with the trained adapter (optional later)
# Load adapter: from peft import PeftModel; chex.model = PeftModel.from_pretrained(chex.model, "chexagent-lora")
